# Delitos informáticos en Colombia — pipeline de datos (fases 2 a 8)

**Curso:** Gestión de Datos Masivos · G002 · Unidad 3
**Integrante:** Alejandro Ramírez
**Fuente:** [DELITOS INFORMÁTICOS — datos.gov.co](https://www.datos.gov.co/Seguridad-y-Defensa/DELITOS-INFORM-TICOS/4v6r-wu98/about_data) (MinDefensa, CC BY-SA 4.0)

**Pregunta:** ¿cómo evolucionan los delitos informáticos en Colombia entre 2023 y 2026 y en qué conductas y territorios se concentran?

Este notebook **muestra el resultado de cada fase ejecutada** por los scripts de `src/`
(que son los que corren dentro de Docker). Se puede volver a ejecutar después de
`bash run_pipeline.sh` y lee las mismas zonas: HDFS, PostgreSQL y los reportes de
`evidencias/`.


## Preparación

Todo corre dentro del contenedor `spark` (PySpark + clientes). El notebook se ejecuta con
`make notebook`.


In [1]:
import json, os, sys
base = os.path.abspath("..") if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
os.chdir(base)
sys.path.insert(0, base)

import pandas as pd
from IPython.display import display

from src import config, hdfs_client
from src.common import leer_json, pg_connect

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 160)

print("Directorio de trabajo:", os.getcwd())
print("Ventana analizada:", config.DESDE, "→", config.HASTA or "hoy")
print("API:", config.API_URL)


Directorio de trabajo: /workspace
Ventana analizada: 2023-11-01T00:00:00 → hoy
API: https://www.datos.gov.co/resource/4v6r-wu98.json


## Fase 2 — Ingesta a la zona raw

Se descarga con paginación de 50.000, orden estable `:id` y reintentos exponenciales.
Cada registro viaja en un sobre JSON con `fuente`, `dataset`, `id_registro`,
`fecha_ingesta` y el `payload` original intacto. Los archivos se particionan por
`fecha_ingesta=AAAA-MM-DD` y cada corrida deja un manifiesto con URL, parámetros y conteo.


In [2]:
from pathlib import Path

rutas = sorted(Path("data/raw/_manifests").glob("manifest_*.json"))
corridas = []
for r in rutas:
    m = leer_json(str(r))
    corridas.append({
        "run_id": m["run_id"],
        "registros": m["registros"],
        "paginas": m["paginas"],
        "bytes": m["bytes"],
        "duracion_seg": m["duracion_seg"],
        "watermark_anterior": m["watermark_anterior"],
    })
display(pd.DataFrame(corridas))

manifiesto = max((leer_json(str(r)) for r in rutas), key=lambda m: m["registros"])
print("Corrida completa de referencia:", manifiesto["run_id"])
print(json.dumps(manifiesto["parametros"], indent=2, ensure_ascii=False))
print("Archivos:", len(manifiesto["archivos"]))


,run_id,registros,paginas,bytes,duracion_seg,watermark_anterior
0,20261126T235353-39101f,232939,5,118895192,20.30,None
1,20261126T235910-1c0486,130,1,66158,0.74,2026-10-31T00:00:00


Corrida completa de referencia: 20261126T235353-39101f
{
  "$select": ":id,fecha_hecho,cod_depto,departamento,cod_muni,municipio,descripcion_conducta,cantidad",
  "$limit": 50000,
  "$order": ":id",
  "$where": "fecha_hecho >= '2023-11-01T00:00:00'"
}
Archivos: 5


In [3]:
with open("data/muestra_raw.jsonl", encoding="utf-8") as fh:
    primera = json.loads(fh.readline())
print(json.dumps(primera, ensure_ascii=False, indent=2)[:1200])


{
  "fuente": "datos.gov.co",
  "dataset": "4v6r-wu98",
  "id_registro": "row-zier~fukf~frge",
  "id_fuente": "row-zier~fukf~frge",
  "hash_contenido": "d4962937c68fd163cf26f141550b2778c4c9a85b",
  "fecha_ingesta": "2026-11-26T23:53:53.413Z",
  "payload": {
    ":id": "row-zier~fukf~frge",
    "fecha_hecho": "2026-10-31T00:00:00.000",
    "cod_depto": "44",
    "departamento": "LA GUAJIRA",
    "cod_muni": "44035",
    "municipio": "ALBANIA",
    "descripcion_conducta": "ARTICULO 269I. HURTO POR MEDIOS INFORMATICOS Y SEMEJANTES",
    "cantidad": "1"
  }
}


## Fase 3 — Mensajería con Kafka

Tópico `delitos-raw` con **3 particiones**. El productor publica cada línea con
`key = id_registro` y el consumidor del grupo `delitos-hdfs-writer` confirma offsets
después de escribir el lote en HDFS (at-least-once).


In [4]:
from kafka.admin import KafkaAdminClient

admin = KafkaAdminClient(bootstrap_servers=config.KAFKA_BOOTSTRAP, client_id="notebook")
info = admin.describe_topics([config.TOPIC_RAW])[0]
display(pd.DataFrame([{"partición": p["partition"], "líder": p["leader"]} for p in info["partitions"]]))
admin.close()
print(open("evidencias/kafka_consumidor.txt", encoding="utf-8").read())


,partición,líder
0,1,1
1,2,1
2,0,1



Consumer group 'delitos-hdfs-writer' has no active members.

GROUP               TOPIC           PARTITION  CURRENT-OFFSET  LOG-END-OFFSET  LAG             CONSUMER-ID     HOST            CLIENT-ID
delitos-hdfs-writer delitos-raw     0          77402           77402           0               -               -               -
delitos-hdfs-writer delitos-raw     1          77798           77798           0               -               -               -
delitos-hdfs-writer delitos-raw     2          77869           77869           0               -               -               -



## Fase 4 — HDFS y MapReduce frente a Spark

El consumidor guarda un archivo por lote (5.000 mensajes) en
`/data/delitos_informaticos/raw/fecha_ingesta=…`. El job de Hadoop Streaming (mapper y
reducer en Python) cuenta casos por conducta, departamento y año; Spark calcula exactamente
lo mismo sobre la misma entrada y se comparan las 48 claves.


In [5]:
partes = [f for f in hdfs_client.listdir(f"{config.HDFS_MAPREDUCE}/conteo") if f["pathSuffix"].startswith("part-")]
texto = hdfs_client.read_text(f"{config.HDFS_MAPREDUCE}/conteo/{partes[0]['pathSuffix']}")

conductas = []
for linea in texto.splitlines():
    if linea.startswith("CONDUCTA|"):
        clave, valor = linea.split("\t")
        conductas.append((clave.split("|", 1)[1], int(valor)))
df_mr = pd.DataFrame(conductas, columns=["conducta", "casos"]).sort_values("casos", ascending=False)
display(df_mr.head(10))

print("Comparación MapReduce vs Spark:")
display(pd.DataFrame([leer_json("evidencias/mapreduce_vs_spark.json")])[
    ["claves_comparadas", "diferencias", "coincide", "total_mapreduce", "total_spark"]
])


,conducta,casos
8,ARTICULO 269I. HURTO POR MEDIOS INFORMATICOS Y...,128934
0,ARTICULO 269A. ACCESO ABUSIVO A UN SISTEMA INF...,47325
5,ARTICULO 269F. VIOLACION DE DATOS PERSONALES,30029
6,ARTICULO 269G. SUPLANTACION DE SITIOS WEB PARA...,13366
9,ARTICULO 269J. TRANSFERENCIA NO CONSENTIDA DE ...,12384
2,ARTICULO 269C. INTERCEPTACION DE DATOS INFORMA...,2363
1,ARTICULO 269B. OBSTACULIZACION ILEGITIMA DE SI...,967
3,ARTICULO 269D. DAÑO INFORMATICO,918
4,ARTICULO 269E. USO DE SOFTWARE MALICIOSO,596
7,ARTICULO 269H. CIRCUNSTANCIAS DE AGRAVACION PU...,1


Comparación MapReduce vs Spark:


,claves_comparadas,diferencias,coincide,total_mapreduce,total_spark
0,48,0,True,232939,232939


## Fase 5 — Zona curada con PySpark

Se eliminan duplicados por `id_registro`, se corrigen tipos, nulos y formatos, y se escribe
Parquet particionado por `anio`. La partición se justifica por las consultas del tablero y
por evitar miles de carpetas de municipio.


In [6]:
metricas = leer_json("evidencias/curated_metrics.json")
display(pd.DataFrame([metricas]).T.rename(columns={0: "valor"}))

muestra = pd.read_parquet("data/curated_sample.parquet")
print("Muestra curada:", muestra.shape)
display(muestra.dtypes.to_frame("tipo").T)
display(muestra.head(5))


,valor
filas_entrada,233069
filas_salida,232939
descartados,130
duplicados_id_registro,130
filas_con_mismo_contenido_conservadas,120115
nulos_columna_clave,0
cantidad_invalida,0
fecha_invalida,0
codigo_invalido,0
particion,anio


Muestra curada: (2000, 14)


,id_registro,hash_contenido,fecha_hecho,fecha_ingesta,anio,mes,anio_mes,cod_depto,departamento,cod_muni,municipio,codigo,conducta,cantidad
tipo,object,object,object,datetime64[ns],int32,int32,object,object,object,object,object,object,object,int32


,id_registro,hash_contenido,fecha_hecho,fecha_ingesta,anio,mes,anio_mes,cod_depto,departamento,cod_muni,municipio,codigo,conducta,cantidad
0,row-2ys4~e27j~f58z,24df705fa214b51ad73628ec1ffcf117195ac7d4,2026-10-31,2026-11-26 23:53:53.413,2026,10,2026-10,50,META,50001,VILLAVICENCIO,269A,ARTICULO 269A. ACCESO ABUSIVO A UN SISTEMA INF...,1
1,row-46r5.6sj8-zpyq,78e7ccf4cc0071d2abdfe4b146680849598ce7bb,2026-10-31,2026-11-26 23:53:53.413,2026,10,2026-10,70,SUCRE,70708,SAN MARCOS,269A,ARTICULO 269A. ACCESO ABUSIVO A UN SISTEMA INF...,1
2,row-23sf-uv9s~x3a4,0891d9d1142875214bf7afdfdbe0fa50dcae2098,2026-10-31,2026-11-26 23:53:53.413,2026,10,2026-10,15,BOYACA,15001,TUNJA,269I,ARTICULO 269I. HURTO POR MEDIOS INFORMATICOS Y...,1
3,row-4gwy~i6g2-r2qu,82e3d6848a3440db8bc30bd2f03baccb8ed0ffc8,2026-10-31,2026-11-26 23:53:53.413,2026,10,2026-10,11,BOGOTA D.C.,11001,BOGOTA D.C.,269J,ARTICULO 269J. TRANSFERENCIA NO CONSENTIDA DE ...,1
4,row-276t_2ics_grih,f81d0d6b78637b438a14fe07cf0f8fe3184131a3,2026-10-31,2026-11-26 23:53:53.413,2026,10,2026-10,73,TOLIMA,73001,IBAGUE,269A,ARTICULO 269A. ACCESO ABUSIVO A UN SISTEMA INF...,1


In [7]:
print("Bytes lógicos en HDFS")
print("  raw JSONL :", f"{hdfs_client.du(config.HDFS_RAW):,}")
print("  Parquet   :", f"{hdfs_client.du(config.HDFS_CURATED):,}")
print("  reducción :", metricas["reduccion_parquet_pct"], "%")


Bytes lógicos en HDFS
  raw JSONL : 118,963,680
  Parquet   : 12,290,007
  reducción : 89.67 %


## Fase 6 — Calidad, marca de agua e idempotencia

Siete validaciones automáticas (tres críticas detienen el pipeline) y un reporte con los
registros descartados. La marca de agua guarda la última fecha cargada; la segunda corrida
sólo trae lo nuevo (más el último día, para correcciones) y la curaduría deduplica.


In [8]:
reporte = leer_json("evidencias/calidad_reporte.json")
print("Estado:", reporte["estado"], "· filas curadas:", f"{reporte['filas_curadas']:,}")
display(pd.DataFrame(reporte["checks"])[["nombre", "critico", "fallos", "descripcion"]])

print("\nCorridas registradas (idempotencia):")
display(pd.DataFrame(leer_json("evidencias/idempotencia_corridas.json")))


Estado: ok · filas curadas: 232,939


,nombre,critico,fallos,descripcion
0,conteo_mayor_cero,True,0,La zona curada tiene al menos un registro
1,sin_nulos_columnas_clave,True,0,"Sin nulos en fecha_hecho, cod_muni, conducta y..."
2,sin_id_registro_repetido,True,0,id_registro único en toda la zona curada
3,cantidad_en_rango,False,0,cantidad entre 1 y 1000
4,fechas_en_rango,False,0,fecha_hecho entre 2000-01-01 y hoy
5,codigos_dane_validos,False,0,cod_depto de 2 dígitos y cod_muni de 5
6,sin_conductas_vacias,False,0,Toda fila tiene descripción de conducta y códi...



Corridas registradas (idempotencia):


,etiqueta,fecha,run_id,registros_ingestados_api,watermark_anterior,filas_raw_acumuladas,duplicados_eliminados,filas_curadas,casos_totales,filas_zona_servicio,filas_stream,watermark_nueva,estado_calidad
0,pipeline-completo,2026-11-26T23:59:10,20261126T235353-39101f,232939,None,232939,0,232939,236883,232939,669,2026-10-31 00:00:00,ok
1,segunda-corrida-idempotencia,2026-11-27T00:00:59,20261126T235910-1c0486,130,2026-10-31T00:00:00,233069,130,232939,236883,232939,669,2026-10-31 00:00:00,ok


## Fase 7 — Capa rápida con Structured Streaming

El flujo lee `delitos-stream` (últimos 90 días), agrega por ventana diaria y artículo con
watermark de 120 días y checkpoint en HDFS, y hace `upsert` en `service.stream_diario`.
Al reiniciar continúa desde los offsets confirmados: no reprocesa ni duplica.


In [9]:
with pg_connect() as conn:
    stream = pd.read_sql("SELECT * FROM service.stream_diario ORDER BY fecha, codigo", conn)

print("Filas en la capa rápida:", len(stream), "· casos:", f"{int(stream['casos'].sum()):,}")
display(stream.head(8))
print(open("evidencias/streaming_reinicio.md", encoding="utf-8").read())


Filas en la capa rápida: 669 · casos: 21,371


/tmp/ipykernel_4353/3878083025.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  stream = pd.read_sql("SELECT * FROM service.stream_diario ORDER BY fecha, codigo", conn)


,fecha,codigo,conducta,casos,actualizado
0,2026-08-02,269A,ARTICULO 269A. ACCESO ABUSIVO A UN SISTEMA INF...,39,2026-11-26 23:58:11.968605
1,2026-08-02,269C,ARTICULO 269C. INTERCEPTACION DE DATOS INFORMA...,1,2026-11-26 23:58:11.968605
2,2026-08-02,269E,ARTICULO 269E. USO DE SOFTWARE MALICIOSO,1,2026-11-26 23:58:11.968605
3,2026-08-02,269F,ARTICULO 269F. VIOLACION DE DATOS PERSONALES,19,2026-11-26 23:58:11.968605
4,2026-08-02,269G,ARTICULO 269G. SUPLANTACION DE SITIOS WEB PARA...,9,2026-11-26 23:58:11.968605
5,2026-08-02,269I,ARTICULO 269I. HURTO POR MEDIOS INFORMATICOS Y...,114,2026-11-26 23:58:11.968605
6,2026-08-02,269J,ARTICULO 269J. TRANSFERENCIA NO CONSENTIDA DE ...,17,2026-11-26 23:58:11.968605
7,2026-08-03,269A,ARTICULO 269A. ACCESO ABUSIVO A UN SISTEMA INF...,57,2026-11-26 23:58:11.968605


# Streaming: reinicio e idempotencia

## Corridas del flujo

| Corrida | Offset final | Filas en servicio | Casos |
|---|---|---:|---:|
| 2026-11-26T23:58:20 | `{'delitos-stream': {'2': 7154, '1': 7030, '0': 7187}}` | 669 | 21371 |
| 2026-11-26T23:58:54 | `{'delitos-stream': {'2': 7154, '1': 7030, '0': 7187}}` | 669 | 21371 |

La segunda corrida reutiliza el checkpoint de HDFS: continúa desde los offsets confirmados y el `upsert` por `(fecha, codigo)` deja el mismo número de filas y casos. No se reprocesa ni se duplica.

## Capa rápida vs capa batch (mismas fechas)

- Casos en streaming: **21,371**
- Casos en batch: **21,371**
- Filas por (fecha, artículo): stream=669, batch=669
- Diferencias: **0**

Las dos capas coinciden: el flujo diario reproduce exactamente el agregado batch de la ventana reciente (arquitectura Lambda).



## Fase 8 — Zona de servicio y tablero

PostgreSQL guarda los indicadores ya calculados: el tablero sólo consulta. Estas son las
tablas que alimentan las 5 pestañas de Streamlit.


In [10]:
consultas = {
    "KPIs": "SELECT * FROM service.ind_kpis",
    "Casos por año": "SELECT * FROM service.ind_casos_anio ORDER BY anio",
    "Conductas (top 5)": "SELECT codigo, SUM(casos) AS casos FROM service.ind_conducta_anio GROUP BY codigo ORDER BY casos DESC LIMIT 5",
    "Departamentos (top 5)": "SELECT departamento, SUM(casos) AS casos FROM service.ind_departamento_anio GROUP BY departamento ORDER BY casos DESC LIMIT 5",
    "Municipios (top 5)": "SELECT municipio, departamento, casos, ranking FROM service.ind_municipio ORDER BY ranking LIMIT 5",
}
with pg_connect() as conn:
    for titulo, sql in consultas.items():
        print(f"\n### {titulo}")
        display(pd.read_sql(sql, conn))



### KPIs


/tmp/ipykernel_4353/829419065.py:11: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  display(pd.read_sql(sql, conn))


,id,total_casos,total_registros,conducta_top,casos_conducta_top,depto_top,casos_depto_top,anio_pico,casos_anio_pico,fecha_corte,actualizado
0,1,236883,232939,269I - ARTICULO 269I. HURTO POR MEDIOS INFORMA...,128934,BOGOTA D.C.,72851,2026,81336,2026-10-31,2026-11-27 00:00:37.885370



### Casos por año


/tmp/ipykernel_4353/829419065.py:11: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  display(pd.read_sql(sql, conn))


,anio,casos,registros,participacion
0,2023,11374,11374,0.0480
1,2024,74845,72996,0.3160
2,2025,69328,67233,0.2927
3,2026,81336,81336,0.3434


... (5 outputs left out)

,departamento,casos
0,BOGOTA D.C.,72851.0
1,ANTIOQUIA,32756.0
2,VALLE DEL CAUCA,20919.0
3,CUNDINAMARCA,15636.0
4,ATLANTICO,10948.0



### Municipios (top 5)


/tmp/ipykernel_4353/829419065.py:11: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  display(pd.read_sql(sql, conn))


,municipio,departamento,casos,ranking
0,BOGOTA D.C.,BOGOTA D.C.,72851,1
1,MEDELLIN,ANTIOQUIA,18709,2
2,CALI,VALLE DEL CAUCA,13019,3
3,BARRANQUILLA,ATLANTICO,8274,4
4,CARTAGENA DE INDIAS,BOLIVAR,5110,5


## Conclusiones

- **232.939 registros** (236.883 casos) entre 2023-11 y 2026-10, con crecimiento sostenido
  y un pico de 81.336 casos en lo que va de 2026.
- **Hurto por medios informáticos (269I)** concentra el 54,4 % de los casos; le siguen
  acceso abusivo (269A) y violación de datos personales (269F).
- **Bogotá, Antioquia y Valle del Cauca** reúnen el 53,4 % de los casos.
- El pipeline es **idempotente**: una segunda corrida reingresó 130 filas, eliminó 130
  duplicados y dejó los mismos 232.939 registros y 236.883 casos.
- **MapReduce y Spark coinciden** en las 48 claves comparadas y el streaming reproduce el
  agregado batch de la ventana reciente (0 diferencias).

Para levantar todo: `docker compose up -d --wait && bash run_pipeline.sh` · Tablero en
<http://localhost:8501>.
